<a href="https://colab.research.google.com/github/tuankhoin/CO3005-PPL/blob/main/Week_8_JVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ho Chi Minh University of Technology (HCMUT)

CO3005 - Principles of Programming Languages

# Week 8 - Java Virtual Machine/Jasmin

<img src="https://i.programmerhumor.io/2024/01/programmerhumor-io-java-memes-backend-memes-f561233f7798cb8.jpg" height=300/>

---
# 1. Compiler Pipeline

A typical compiler works as follows:

* Scanner → Parser → Semantic Analysis
* Build AST (Abstract Syntax Tree)
* Code Generation
* Jasmin (`.j`) → Bytecode (`.class`) → JVM execution

Jasmin is the **bridge between compiler and JVM**

---

# 2. Why JVM?

JVM enables:

* Platform independence
* Same bytecode runs everywhere

```text
Compile once → run anywhere
```

👉 Bytecode = portable intermediate representation

---

# 3. JVM = Stack-Based Machine

JVM does NOT use registers.

Instead:

* Uses an **operand stack**
* All operations happen on stack values

## Execution Model

![](https://i.imgur.com/Kv9ichJ.gif)

```text
push operands → apply operation → push result
```

Example:

```text
b * 2 + 40
```

becomes:

```text
push b → push 2 → mul → push 40 → add
```

---


# 4. JVM Runtime Structure

Main components:

* Method Area → class metadata
* Heap → objects
* Stack → execution frames
* PC Register → current instruction

Each method call creates a **stack frame**

---


# 5. JVM Data Types & Naming

## 5.1 Type Descriptors

Used in method signatures:

| Type    | Descriptor |
| ------- | ---------- |
| int     | I          |
| float   | F          |
| double  | D          |
| long    | J          |
| char    | C          |
| boolean | Z          |
| void    | V          |

## 5.2 Reference Types

```text
L<full-class-name>;
```

Example:

```text
Ljava/lang/String;
```

Means: object of class `java.lang.String`

## 5.3 Array Types

```text
[<type>
```

Examples:

```text
[I       → int[]
[[I      → int[][]
[Ljava/lang/String; → String[]
```

👉 Rule:

```text
number of '[' = number of dimensions
```

## 5.4 Method Descriptors

Format:

```text
(param_type_1 param_type_2...) return_type
```
Example:

```text
int f(int, int)

Descriptor:
(II)I
```

Breakdown:

* `(II)` → parameters: int, int
* `I` → return: int

More Examples

```text
(F)I
→ int f(float)
```

```text
([I)V
→ void f(int[])
```

```text
([Ljava/lang/String;)V
→ void main(String[])
```

```text
(I[F)D
→ double f(int, float[])
```

## 5.5 How JVM Uses This

Example:

```text
invokevirtual java/io/PrintStream/println(I)V
```

Breakdown:

* method name: println
* `(I)` → takes int
* `V` → returns void



---

# 6. Operand Stack (Core Concept)

## 6.1 Type Prefix Convention

JVM encodes types directly in instruction names:

| Prefix | Type                            |
| ------ | ------------------------------- |
| i      | int (also boolean, char, short) |
| f      | float                           |
| d      | double                          |
| l      | long                            |
| a      | reference                       |

Naming Pattern:

```text
[type] + operation + optional detail
```

Examples:

```text
iadd     → int addition
fload_2  → load float from slot 2
iaload   → load int from array
```

The stack is used for:

* operands
* intermediate results
* arguments
* return values

---

## Example

```text
a = b * 2 + 40
```

Jasmin:

```text
iload_2
iconst_2
imul
bipush 40
iadd
istore_1
```

## How to Read Instructions

```text
iload_2
```

means:

* `i` → int
* `load` → load variable
* `2` → local slot index

👉 load `int` from local variable 2 onto stack

More details and conventions below!

---


# 7. Local Variables

* Stored in indexed slots
* Indexed from 0

Rules:

* Static method → slot 0 = first parameter
* Instance method → slot 0 = this

## Load / Store Pattern

```text
[type]load / [type]store
```

Examples:

```text
iload, istore
fload, fstore
aload, astore
```

Shortcut:

```text
iload_0, iload_1, ...
```

---

# 8. Constants & Stack Ops

## Constants

```text
iconst_0 → push 0
iconst_5 → push 5
fconst_1 → push 1.0
```

Other:

* bipush → small int
* sipush → medium int
* ldc → general constant

## Stack Operations

* pop → remove top
* dup → duplicate top

---

# 9. Arithmetic & Operations

Pattern:

```text
[type] + operation
```

Examples:

```text
iadd → +
isub → -
imul → *
idiv → /
```

---

## Comparison Keywords

| Keyword | Meaning |
| ------- | ------- |
| eq      | ==      |
| ne      | !=      |
| lt      | <       |
| le      | ≤       |
| gt      | >       |
| ge      | ≥       |

---

# 10. Control Flow (COMPLETE)

Control flow in JVM consists of:

* Conditional branches
* Unconditional jumps
* Switch (multi-branch)
* Return instructions
* Structural directives


## 10.1 Conditional Branches

### Integer comparison

```text
if_icmp[cond]
```

Examples:

```text
if_icmpeq → ==
if_icmpgt → >
```


### Single value comparison

```text
if[cond]
```

Examples:

```text
ifeq → == 0
ifgt → > 0
```


## 10.2 Floating-Point Comparison

JVM does NOT support direct float branching.

Instead:

```text
fcmpl / fcmpg → then if...
```


### Behavior

```text
[value1, value2]
→ fcmpl
→ result (-1, 0, 1)
→ ifgt / iflt ...
```

### Difference

| Instruction | NaN behavior |
| ----------- | ------------ |
| fcmpl       | returns -1   |
| fcmpg       | returns +1   |

### Example

```text
fload a
fload b
fcmpl
ifgt L1
```

## 10.3 Unconditional Jump

```text
goto Label
```
Always jumps

## 10.4 Switch Instructions

### `tableswitch` (dense values)

* Index: Labels only
* Efficient for continuous ranges
* Still create stuffs for a non-existent key if it is within the range.

### `lookupswitch` (sparse values)

* Index: Keys and Labels
* Efficient for scattered values

## 10.5 Return Instructions

| Instruction | Meaning       |
| ----------- | ------------- |
| ireturn     | return int    |
| freturn     | return float  |
| areturn     | return object |
| return      | void          |

## 10.6 Jasmin Directives (Structure)

Not runtime instructions:

```text
.method
.limit stack
.limit locals
.end method
```






---

# 11. Arrays

Pattern:

```text
[type]aload / [type]astore
```

Examples:

```text
iaload → load int from array
iastore → store int into array
aaload → load object reference
```

---

# 12. Method Invocation

Types:

* invokestatic
* invokevirtual
* invokespecial

## Reading Invocation

```text
invokevirtual java/io/PrintStream/println(Ljava/lang/String;)V
```

Breakdown:

* class: PrintStream
* method: println
* parameter: String
* return: void

---

# 13. Jasmin File Structure

```text
.source File.java
.class public ClassName
.super java/lang/Object

.method ...
.limit stack N
.limit locals M
...
.end method
```

Must define stack + locals explicitly

---

# 14. Code Generation Overview

3 levels:

* Machine-independent (AST-based)
* Intermediate (stack simulation)
* Machine-dependent (emit Jasmin)

---

# 15. Emitter (Instruction Generator)

Naming pattern:

```text
emit + OPERATION
```

Examples:

```text
emitADDOP → iadd / fadd
emitMULOP → imul / fmul
emitRELOP → if_icmp...
emitILOAD → iload
```

emit = generate instructions

---

# 16. Frame (Execution Simulation)

Tracks:

* Labels (if, loop, break)
* Local variables
* Operand stack

Functions:

```text
push(), pop()
getNewIndex()
getMaxOpStackSize()
```

---

# 17. Expression Code Generation

Rule:

```text
Left → Right → Operator
```

Example:

```text
(a + b) * 3
```

Jasmin:

```text
iload a
iload b
iadd
iconst_3
imul
```

👉 This is **post-order traversal of AST**

---

# 18. Loop Pattern

Example:

```text
while (i < 10)
```

Jasmin:

```text
L0:
iload i
bipush 10
if_icmpge L1

... body ...

goto L0
L1:
```

---

# 19. Mapping Source → JVM

| Source       | JVM          |
| ------------ | ------------ |
| variable     | local slot   |
| function     | method       |
| global       | static field |
| expression   | stack ops    |
| control flow | jumps        |




---

# 20. Jasmin examples

## Example 1

```java
int a = 1 ;
int b = 100;
int c = 1000;
int d = 40000;
int e = a * b + c – d;
```

```
.line 6
  iconst_1
  istore_1
.line 7
  bipush 100
  istore_2
.line 8
  sipush 1000
  istore_3
.line 9
  ldc 40000
  istore 4
.line 10
  iload_1
  iload_2
  imul
  iload_3
  iadd
  iload 4
  isub
  istore 5
```

## Example 2

```java
a[0] = 100;
b = a[1];
```

```
.line 8
  aload_0 // push address of a
  iconst_0 // push 0
  bipush 100 // push 100
  iastore // a[0] = 100
.line 9
  aload_0 // push address of a
  iconst_1 // push 1
  iaload // pop a and 1, push a[1]
  istore_1 // store to b
```

## Example 3

```java
float a,b; int c;
if (a > b)
  c = 1;
else
  c = 2;
```

```
.line 7
 fload_0 // push a
 fload_1 // push b
 fcmpl // pop a,b, push 1 if a > b, 0 otherwise
 ifle Label0 // goto Label0 if top <= 0
.line 8
 iconst_1
 istore_2
 goto Label1
Label0:
.line 10
 iconst_2
 istore_2
Label1:
```

## Example 4

```java
int a,b,c;
a = b = c = 1;
```

```
.line 7
  iconst_1
  dup
  istore_2
  dup
  istore_1
  istore_0
```

## Example 5

```java
public class VD13 {
  public static void main(String[] arg) {
    goo(new VD13());
  }
  float foo(int a, float b) {
    return a + b;
  }
  static void goo(VD13 x){
    x.foo(1,2.3F);
  }
}
```

```
method public static main([Ljava/lang/String;)V
.limit stack 2
.limit locals 1
.var 0 is arg0 [Ljava/lang/String; from Label0 to Label1
.line 3
  new VD13
  dup
  invokespecial VD13/<init>()V
  invokestatic VD13/goo(LVD13;)V
.line 4
  return
.end method

.method static goo(LVD13;)V
.limit stack 3
.limit locals 1
.var 0 is arg0 LVD13; from Label0 to Label1

.line 9
  aload_0
  iconst_1
  ldc 2.3
  invokevirtual VD13/foo(IF)F
  pop

Label1:
  .line 10

return
.end method

.method foo(IF)F
.limit stack 2
.limit locals 3
.var 0 is this LVD13; from Label0 to Label1
.var 1 is arg0 I from Label0 to Label1
.var 2 is arg1 F from Label0 to Label1

Label0:
  iload_1
  i2f
  fload_2
  fadd

Label1:
  freturn

.end method
```

---
# Step-by-step Conversion (Example 5)
---

# 1. Method descriptors

We have 3 methods: main, goo, foo ⇒ Jasmin needs 3 `.method` blocks.

Use the rule:

* `int` → `I`
* `float` → `F`
* `void` → `V`
* `String[]` → `[Ljava/lang/String;`
* object `VD13` → `LVD13;`

So:

### `main(String[] arg): void`

```text
([Ljava/lang/String;)V
```

### `goo(VD13 x): void`

```text
(LVD13;)V
```

### `foo(int a, float b): float`

```text
(IF)F
```

Thus the method headers are:

```text
.method public static main([Ljava/lang/String;)V
.method static goo(LVD13;)V
.method foo(IF)F
```

---

# 2. Assign local variable slots

## `main` is static

Static methods do **not** have `this`.

So:

* slot 0 = `arg`

Hence:

```text
.limit locals 1
.var 0 is arg0 [Ljava/lang/String; from Label0 to Label1
```

## `goo` is static

* slot 0 = parameter `x`

So:

```text
.limit locals 1
.var 0 is arg0 LVD13; from Label0 to Label1
```

## `foo` is an instance method

Instance methods reserve slot 0 for `this`.

So:

* slot 0 = `this`
* slot 1 = `a`
* slot 2 = `b`

Thus:

```text
.limit locals 3
.var 0 is this LVD13; from Label0 to Label1
.var 1 is arg0 I from Label0 to Label1
.var 2 is arg1 F from Label0 to Label1
```

---

# 4. Translate `main`

Java:

```java
goo(new VD13());
```

We must:

1. create a new object
2. initialize it
3. pass it to static method `goo`

## Step 4.1 Create object

```text
new VD13
```

Pushes an uninitialized object reference onto the stack.

Stack:

```text
[obj]
```

## Step 4.2 Duplicate it

```text
dup
```

Why? Because `invokespecial <init>` will consume one copy, but we still need one copy to pass into `goo`.

Stack:

```text
[obj, obj]
```

## Step 4.3 Call constructor

```text
invokespecial VD13/<init>()V
```

Consumes one object reference and initializes it.

Stack after call:

```text
[obj]
```

## Step 4.4 Call static method

```text
invokestatic VD13/goo(LVD13;)V
```

Consumes the remaining object reference as argument to `goo`.

Stack:

```text
[]
```

## Step 4.5 Return

```text
return
```

### Why `.limit stack 2`?

Maximum stack depth is here:

```text
new VD13   -> 1
dup        -> 2
```

So max = 2.

---

# 5. Translate `goo`

Java:

```java
x.foo(1,2.3F);
```

This is an **instance method call**, so JVM needs:

1. object reference `x`
2. first arg `1`
3. second arg `2.3`

Then call `invokevirtual`.

## Step 5.1 Load `x`

`x` is in slot 0:

```text
aload_0
```

Stack:

```text
[x]
```

## Step 5.2 Push `1`

```text
iconst_1
```

Stack:

```text
[x, 1]
```

## Step 5.3 Push `2.3F`

```text
ldc 2.3
```

Stack:

```text
[x, 1, 2.3]
```

## Step 5.4 Call method

```text
invokevirtual VD13/foo(IF)F
```

Consumes:

```text
x, 1, 2.3
```

and pushes result of type `float`.

Stack after call:

```text
[result]
```

## Step 5.5 Discard result

Because Java statement:

```java
x.foo(1,2.3F);
```

does not use the returned float.

So:

```text
pop
```

Stack:

```text
[]
```

## Step 5.6 Return

```text
return
```

### Why `.limit stack 3`?

Maximum stack depth occurs before `invokevirtual`:

```text
[x, 1, 2.3]
```

So max = 3.

---

# 6. Translate `foo`

Java:

```java
return a + b;
```

But `a` is `int` and `b` is `float`, so we must convert `a` to float first.

## Step 6.1 Load `a`

`a` is slot 1:

```text
iload_1
```

Stack:

```text
[a]
```

## Step 6.2 Convert int to float

```text
i2f
```

Stack:

```text
[(float)a]
```

## Step 6.3 Load `b`

`b` is slot 2:

```text
fload_2
```

Stack:

```text
[(float)a, b]
```

## Step 6.4 Add floats

```text
fadd
```

Stack:

```text
[result]
```

## Step 6.5 Return float

```text
freturn
```

### Why `.limit stack 2`?

Max stack depth is: `[(float)a, b]` so max = 2.




---

# Final Summary

* JVM = stack-based execution
* Jasmin = readable bytecode
* Naming is structured and systematic
* Control flow = labels + jumps
* Float comparison = 2-step process
* Code generation = simulate stack + emit instructions